In [1]:
import numpy as np
import pandas as pd


In [2]:
directory = 'data/dirty_cafe_sales.csv'


In [3]:
# Clean load
na_vals = ['UNKNOWN', 'ERROR', 'nan', 'NaN', '']
df = pd.read_csv(directory, na_values=na_vals)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# Check duplicates and exact missing percentages
print(f"Duplicates: {df.duplicated().sum()}")
missing_summary = pd.DataFrame({
    'dtype': df.dtypes,
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'unique_vals': df.nunique()
})
print(missing_summary)

Duplicates: 0
                    dtype  missing_count  missing_pct  unique_vals
transaction_id     object              0         0.00        10000
item               object            969         9.69            8
quantity          float64            479         4.79            5
price_per_unit    float64            533         5.33            6
total_spent       float64            502         5.02           17
payment_method     object           3178        31.78            3
location           object           3961        39.61            2
transaction_date   object            460         4.60          365


Cleaning the File


In [4]:
# 1. Convert types properly (keeping NaNs intact!)
num_cols = ['quantity', 'price_per_unit', 'total_spent']
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')

# 2. Recover missing Price Per Unit from known Item prices (and vice versa)
item_price_map = df.dropna(subset=['item', 'price_per_unit']).groupby('item')['price_per_unit'].median()
df['price_per_unit'] = df['price_per_unit'].fillna(df['item'].map(item_price_map))


# 3. Recover missing math fields using Total = Quantity * Price
df['total_spent'] = df['total_spent'].fillna(df['quantity'] * df['price_per_unit'])
df['quantity'] = df['quantity'].fillna(df['total_spent'] / df['price_per_unit'])
df['price_per_unit'] = df['price_per_unit'].fillna(df['total_spent'] / df['quantity'])

# 4. For categorical columns with high missingness (like location), impute a category rather than dropping 40% of your data
df['location'] = df['location'].fillna('Not Recorded')
df['payment_method'] = df['payment_method'].fillna('Not Recorded')


In [5]:
# 1. Group by price and get the unique items for each price
price_to_items = df.dropna(subset=['item', 'price_per_unit']).groupby('price_per_unit')['item'].unique()

# 2. Build a mapping dictionary ONLY for 1-to-1 relationships (where a price maps to exactly one unique item)
price_to_single_item = {
    price: items[0]
    for price, items in price_to_items.items()
    if len(items) == 1
}

# 3. Use map with the clean scalar-valued dictionary and apply fillna
df['item'] = df['item'].fillna(df['price_per_unit'].map(price_to_single_item))

# 4. turn item, transaction_id columns into string
df['item'] = df['item'].astype('string')
df['transaction_id'] = df['transaction_id'].astype('string')
df['location'] = df['location'].astype('string')
df['payment_method'] = df['payment_method'].astype('string')


# 5. making the TXN_ID to just ID
df['transaction_id'] = df['transaction_id'].str.replace('TXN_', '', regex=False).astype('int64')

In [6]:
#check errors withing the code for total amount
math_mismatch = df[np.abs((df['quantity']*df['price_per_unit']) - df['total_spent']) > 1e-5]
print(f"Math errors in total spent: {len(math_mismatch)}")

#fill unknown items with 'unknown items'
df['item'] = df['item'].fillna('Unknown Item')

Math errors in total spent: 0


In [7]:
df.isna().sum()

,0
transaction_id,0
item,0
quantity,23
price_per_unit,6
total_spent,23
payment_method,0
location,0
transaction_date,460


In [8]:
#checking the NaN values of quantity column
df[df['quantity'].isna()]
#dropping the NaN entries
df = df.dropna(subset=['quantity', 'total_spent'])
df['quantity'] = df['quantity'].astype('int')

the dataset is clean now, it can be used for further analysis.



In [9]:
print(df.dtypes)
print(df.describe())
df.to_csv('clean_coffee_sales.csv', index=False)

transaction_id               int64
item                string[python]
quantity                     int64
price_per_unit             float64
total_spent                float64
payment_method      string[python]
location            string[python]
transaction_date    datetime64[ns]
dtype: object
       transaction_id     quantity  price_per_unit  total_spent  \
count    9.974000e+03  9974.000000     9974.000000  9974.000000   
mean     5.474370e+06     3.024865        2.946962     8.927411   
min      1.000555e+06     1.000000        1.000000     1.000000   
25%      3.221999e+06     2.000000        2.000000     4.000000   
50%      5.453526e+06     3.000000        3.000000     8.000000   
75%      7.729895e+06     4.000000        4.000000    12.000000   
max      9.999124e+06     5.000000        5.000000    25.000000   
std      2.594863e+06     1.420504        1.280144     6.002644   

                    transaction_date  
count                           9514  
mean   2023-07-01 23:14: